# JA·LE — Colab Training & Evaluation

**TVS Credit E.P.I.C 8.0 · Problem (e) Swarm Intelligence Lending Network**

This notebook is the full-quality version of the JA·LE pipeline. The sandbox V1 (scipy + scikit-learn, 18 s, verified) is reproduced here as a **sanity baseline**, then extended with the parts that could not run there:

1. **PyTorch Geometric GNNs** — GraphSAGE, GAT, GCN, CARE-GNN (simplified), BWGNN
2. **Real public benchmarks** — YelpChi, Amazon, T-Finance, DGraph-Fin
3. **Scale** — the FULL profile (120,000 persons)

---
### What is verified and what is not — read this first

| Component | Status |
|---|---|
| Synthetic generator, entity resolution, graph + feature builder, sklearn baselines, all leakage audits | **Executed and verified** in the sandbox. Numbers in `reports/v1_SMOKE.json`. |
| `jale/models/torch_gnn.py` | **Never executed.** No torch in the sandbox. Syntax-checked only. Expect shape errors on the first run — that is a known gap, not a surprise. |
| `jale/data/public_datasets.py` | **Never executed.** Same reason. Dataset schemas were written from the papers and dataset cards, not from inspecting the files. |

If a cell fails, fix it and record what you changed — the point of this notebook is to produce trustworthy numbers, not to look like it ran.


## 0. Environment


In [4]:
# Runtime -> Change runtime type -> GPU (T4) for the GNN section.
# CPU is fine for sections 1-4 and for YelpChi.
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'torch_geometric', 'pyarrow'])

import torch
print('torch        :', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU           :', torch.cuda.get_device_name(0))
import torch_geometric
print('PyG           :', torch_geometric.__version__)


torch        : 2.11.0+cu128
CUDA available: True
GPU           : Tesla T4
PyG           : 2.8.0.post1


## 1. Get the code

Two options. **Option A** (upload) always works. **Option B** (Drive) is better if you are iterating.

**Option A:** zip the `jale/` project directory on your machine, then `Files -> Upload to session storage`, and run the cell below.


In [5]:
import os, zipfile

if os.path.exists('jale.zip'):
    with zipfile.ZipFile('jale.zip') as z: z.extractall('.')
    print('extracted jale.zip')


sys_path_added = os.getcwd()
import sys
if sys_path_added not in sys.path: sys.path.insert(0, sys_path_added)

from jale.config import SMOKE, FULL
print('import OK')


extracted jale.zip
import OK


## 2. Configuration


In [6]:
from jale.config import ObservationTime

PROFILE   = SMOKE          # or FULL for 120k persons (~minutes, more RAM)
OBS_TIME  = ObservationTime.APPLICATION   # no repayment history: the honest setting
N_SPLITS  = 5
SEED      = 0
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'profile={PROFILE.name if hasattr(PROFILE,"name") else PROFILE}  obs={OBS_TIME}  device={DEVICE}')


profile=smoke  obs=application  device=cuda


## 3. Build the synthetic portfolio, graph and features

`build_dataset` writes observables to `raw/` and ground truth to `labels/`. Everything downstream reads **only** `raw/`. The assertion below is the leakage defence: if any ground-truth column ever reaches the feature tables, this fails.


In [7]:
import numpy as np, pandas as pd
from pathlib import Path
from jale.data.generator import build as build_dataset
from jale.graph.builder import build_graph, fold_groups
from jale.features.builder import build_node_features, build_graph_features

root = build_dataset(PROFILE, 'data/jale_colab')
tabs = {f.stem: pd.read_parquet(f) for f in sorted((root/'raw').glob('*.parquet'))}

BANNED = ('ring_id', 'human_id', 'is_kiosk')
for name, df in tabs.items():
    bad = [c for c in df.columns if c.startswith('label') or c in BANNED]
    assert not bad, f'{name} leaks ground truth: {bad}'
print('label-segregation assertion: PASS')

apps = tabs['applications']
lab  = pd.read_parquet(root/'labels'/'application_labels.parquet')
y = (lab.set_index('application_id')['ring_id']
        .reindex(apps['application_id']).fillna(0).to_numpy() > 0).astype(int)
print(f'{len(apps):,} applications | {y.sum():,} fraud ({y.mean():.2%})')


label-segregation assertion: PASS
3,474 applications | 131 fraud (3.77%)


In [8]:
graph = build_graph(apps, tabs['guarantor_links'], tabs['persons'])
A = graph.cooccurrence_union()
groups = fold_groups(graph).to_numpy()
print('relations:', {r: graph.incidence[r].shape[1] for r in graph.relations()})
print(f'graph: {A.shape[0]:,} nodes, {A.nnz:,} directed edges, avg degree {A.nnz/A.shape[0]:.1f}')
print(f'ring-disjoint fold groups: {len(np.unique(groups)):,} (largest {pd.Series(groups).value_counts().max()})')


relations: {'device': 2404, 'dealer': 172, 'account': 3450, 'person': 2678, 'guarantor': 1239}
graph: 3,474 nodes, 511,652 directed edges, avg degree 147.3
ring-disjoint fold groups: 1,983 (largest 116)


In [9]:
nf = build_node_features(apps, tabs['emi_schedule'], OBS_TIME)
gf = build_graph_features(graph, apps, nf)

NODE_COLS = [c for c in nf.columns if not c.endswith(('_freq', '_code'))]
G_COLS    = [c for c in gf.columns if c != 'ppr']
X_node = nf.reindex(graph.app_ids)[NODE_COLS]
X_all  = X_node.join(gf.reindex(graph.app_ids)[G_COLS], how='left')

# Observation-time gate: a repayment-history column must never appear here.
HIST = ('n_missed', 'dpd', 'miss_rate', 'ever_dpd30', 'first_missed')
leaked = [c for c in X_all.columns if any(h in c for h in HIST)]
assert not leaked, f'repayment history leaked: {leaked}'

print(f'node features: {len(NODE_COLS)} | graph features: {len(G_COLS)} | total: {X_all.shape[1]}')
print('observation-time gate: PASS')


node features: 11 | graph features: 75 | total: 86
observation-time gate: PASS


## 4. sklearn baselines under ring-disjoint CV

This reproduces the sandbox numbers. On SMOKE the expected output is:

| Model | AUC-PR | Lift |
|---|---|---|
| best single node feature (`n_guarantors`), raw | 0.233 | 6.2x |
| node-only logistic | 0.144 | 3.8x |
| node-only GBT | 0.243 | 6.5x |
| **node + graph GBT** | **0.754** | **20.0x** |
| graph-regularised LR | 0.423 | 11.2x |

> **Note the first row.** `n_guarantors` alone gives 6.2x lift. An earlier version of
> this project reported node-only GBT at 0.038 (chance) and concluded the generator
> injects no per-applicant signal. That was a `min_samples_leaf=20` bug: with 131
> positives over 5 folds there are ~26 per fold, so a leaf needing 20 samples can
> barely split on the positive class. `best_single_feature()` now runs every time so
> a broken baseline cannot flatter the model again.

If these differ materially, something in the environment changed — investigate
before trusting the GNN numbers that follow.


In [10]:
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from jale.models.models import fit_logistic, fit_gbt, GraphRegularisedLogistic, GBT_DEFAULTS
from jale.eval.metrics import full_metrics, best_single_feature

print('GBT defaults:', GBT_DEFAULTS)   # leaf=10, not 20 -- see the note above

def prep(X, fit_mask):
    Xv = X.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)
    return StandardScaler().fit(Xv[fit_mask]).transform(Xv)

def cv(X, y, groups, kind):
    oof = np.zeros(len(y))
    for tr, te in GroupKFold(n_splits=N_SPLITS).split(X, y, groups=groups):
        m = np.zeros(len(y), dtype=bool); m[tr] = True
        Xtr, Xte = prep(X, m)[tr], prep(X, m)[te]
        if kind == 'lr':      s = fit_logistic(Xtr, y[tr]).decision_function(Xte)
        elif kind == 'gbt':   s = fit_gbt(Xtr, y[tr], seed=SEED).predict_proba(Xte)[:, 1]
        elif kind == 'glr':   s = GraphRegularisedLogistic(lam=0.5).fit(Xtr, y[tr], A[tr][:, tr]).decision_function(Xte)
        oof[te] = s
    return oof

print('\n--- honest floor: best single column used as a raw score ---')
for lbl, X in [('node', X_node), ('graph', gf.reindex(graph.app_ids)[G_COLS])]:
    nm, auc = best_single_feature(y, X)
    print(f'  best single {lbl:5s} feature: {nm:32s} AUC-PR={auc:.4f} lift={auc/y.mean():.1f}x')

print('\n--- models ---')
base = {}
for name, X, kind in [('node-only LR', X_node, 'lr'), ('node-only GBT', X_node, 'gbt'),
                      ('graph-only GBT', gf.reindex(graph.app_ids)[G_COLS], 'gbt'),
                      ('node+graph GBT', X_all, 'gbt'), ('node+graph graph-LR', X_all, 'glr')]:
    m = full_metrics(y, cv(X, y, groups, kind)); base[name] = m
    print(f'{name:20s} AUC-PR={m["auc_pr"]:.4f} lift={m["lift_pr"]:.1f}x AUC-ROC={m["auc_roc"]:.4f} R@5%={m["recall_at_5pct"]:.3f}')


GBT defaults: {'max_depth': 6, 'min_samples_leaf': 10, 'max_iter': 250, 'learning_rate': 0.05, 'l2_regularization': 1.0}

--- honest floor: best single column used as a raw score ---
  best single node  feature: n_guarantors                     AUC-PR=0.2325 lift=6.2x
  best single graph feature: node_deg_guarantor               AUC-PR=0.2891 lift=7.7x

--- models ---
node-only LR         AUC-PR=0.1440 lift=3.8x AUC-ROC=0.6704 R@5%=0.260
node-only GBT        AUC-PR=0.2434 lift=6.5x AUC-ROC=0.5705 R@5%=0.214
graph-only GBT       AUC-PR=0.7511 lift=19.9x AUC-ROC=0.9821 R@5%=0.794
node+graph GBT       AUC-PR=0.7540 lift=20.0x AUC-ROC=0.9798 R@5%=0.733
node+graph graph-LR  AUC-PR=0.4226 lift=11.2x AUC-ROC=0.7301 R@5%=0.420


## 5. PyTorch GNNs on the SAME ring-disjoint folds

This is the comparison that matters. Same graph, same features, same folds, same metric — only the model class changes. Anything else is not a fair comparison.

> **Unverified code.** `jale/models/torch_gnn.py` has never been executed. If a > shape error appears, it is a bug in that file, not in your setup.


In [11]:
from jale.models.torch_gnn import to_pyg_data, build_model, train_and_eval
from jale.graph.builder import STRONG_FOLD_RELATIONS

# TWO fixes here, both of which matter:
#
# 1. The scaler is fitted INSIDE each fold on training rows only. Fitting it once
#    on the whole matrix leaks each test fold's mean and variance into training.
#
# 2. Message passing uses the STRONG relations only (device/account/person/
#    guarantor), NOT the full cooccurrence_union(). Measured: cooccurrence_union()
#    has 511,652 edges of which 403,062 (78.8%) cross fold boundaries, and 98.6% of
#    them come from the dealer relation. A transductive GNN would pass messages
#    from train nodes into test nodes along those edges, destroying the ring-
#    disjoint guarantee. The strong-relations graph has 8,294 edges and ZERO
#    cross-fold edges, so folds are genuinely disconnected components.
#
# Dealer stays in as FEATURES (worth 0.098 AUC-PR, measured) but not as EDGES: at
# 98.6% of all edges it turns the graph into a dense dealer blob that carries no
# fraud structure, which is a large part of why GNNs underperformed GBT.
A_msg = graph.cooccurrence_union(tuple(STRONG_FOLD_RELATIONS))
print(f'message-passing graph: {A_msg.nnz:,} edges (vs {A.nnz:,} in the full union)')

Xraw = X_all.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=np.float32)
y64 = y.astype(np.int64)

results_gnn = {}
fold_iter = list(GroupKFold(n_splits=N_SPLITS).split(Xraw, y64, groups=groups))

for kind in ['sage', 'gat', 'gcn', 'caregnn', 'bwgnn']:
    per_fold = []
    for fi, (tr, te) in enumerate(fold_iter):
        sc = StandardScaler().fit(Xraw[tr])          # train rows only
        Xv = sc.transform(Xraw).astype(np.float32)   # transform all: the GNN needs
        va = te[: len(te) // 2]; te2 = te[len(te) // 2:]   # features for every node
        trm = np.zeros(len(y), bool); trm[tr] = True
        vam = np.zeros(len(y), bool); vam[va] = True
        tem = np.zeros(len(y), bool); tem[te2] = True
        data = to_pyg_data(A_msg, Xv, y64, trm, vam, tem)
        model = build_model(kind, in_dim=Xv.shape[1], hidden=128)
        _, met = train_and_eval(model, data, epochs=150, lr=1e-2, device=DEVICE)
        per_fold.append(met)
    agg = {k: float(np.mean([m[k] for m in per_fold])) for k in per_fold[0]}
    results_gnn[kind] = agg
    print(f'{kind:9s} AUC-PR={agg["auc_pr"]:.4f} AUC-ROC={agg["auc_roc"]:.4f}')


message-passing graph: 8,294 edges (vs 511,652 in the full union)
sage      AUC-PR=0.6880 AUC-ROC=0.8285
gat       AUC-PR=0.6716 AUC-ROC=0.8505
gcn       AUC-PR=0.6701 AUC-ROC=0.8046
caregnn   AUC-PR=0.6699 AUC-ROC=0.8765
bwgnn     AUC-PR=0.0723 AUC-ROC=0.3678


## 6. Leakage audits — run these no matter what

None of the numbers above mean anything unless these pass. A GNN that scores well on the shuffled-label control is memorising, not detecting.


In [12]:
from jale.eval.splits import shuffle_label_control

fold_id = np.zeros(len(y), dtype=int)
for i, (_, te) in enumerate(fold_iter): fold_id[te] = i

yl = shuffle_label_control(y, fold_id, seed=1)
assert float(np.mean(yl)) == float(np.mean(y)), 'control changed the base rate'
print(f'shuffled-label control: base rate preserved at {yl.mean():.4f}')

for name, X in [('node-only GBT', X_node), ('node+graph GBT', X_all)]:
    m = full_metrics(yl, cv(X, yl, groups, 'gbt'))
    ok = m['auc_pr'] < 1.5 * m['base_rate']
    print(f'  {name:16s} AUC-PR={m["auc_pr"]:.4f} vs base {m["base_rate"]:.4f} -> {"PASS" if ok else "FAIL: LEAKAGE"}')


shuffled-label control: base rate preserved at 0.0377
  node-only GBT    AUC-PR=0.0392 vs base 0.0377 -> PASS
  node+graph GBT   AUC-PR=0.0413 vs base 0.0377 -> PASS


In [13]:
# Random split vs ring-disjoint: quantifies how much a ring leaks into a naive eval.
from sklearn.model_selection import StratifiedKFold
oof = np.zeros(len(y))
for tr, te in StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED).split(X_all, y):
    m = np.zeros(len(y), bool); m[tr] = True
    Xtr, Xte = prep(X_all, m)[tr], prep(X_all, m)[te]
    oof[te] = fit_gbt(Xtr, y[tr], seed=SEED).predict_proba(Xte)[:, 1]
leaky = full_metrics(y, oof)
print(f'random split      AUC-PR = {leaky["auc_pr"]:.4f}')
print(f'ring-disjoint     AUC-PR = {base["node+graph GBT"]["auc_pr"]:.4f}')
print(f'leakage gap              = {leaky["auc_pr"] - base["node+graph GBT"]["auc_pr"]:+.4f}')


random split      AUC-PR = 0.8537
ring-disjoint     AUC-PR = 0.7540
leakage gap              = +0.0997


## 6b. Nested CV — the number that goes in the report

Choosing hyperparameters on the same folds used for reporting is selection on the
test set. On SMOKE it is worth **+0.037 AUC-PR**. This cell removes it: selection
happens inside each outer training split only, so the outer test fold is never seen.

Slow (~11 min on SMOKE, longer on FULL). **This is the headline number.**


In [14]:
from jale.models.models import select_gbt

nested = {}
KEYS = {'node-only': 'node-only GBT', 'graph-only': 'graph-only GBT',
        'node+graph': 'node+graph GBT'}
for lbl, X in [('node-only', X_node), ('graph-only', gf.reindex(graph.app_ids)[G_COLS]),
               ('node+graph', X_all)]:
    Xv = X.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    oof = np.zeros(len(y))
    for tr, te in GroupKFold(n_splits=N_SPLITS).split(X, y, groups=groups):
        params, _ = select_gbt(Xv[tr], y[tr], groups[tr])
        sc = StandardScaler().fit(Xv[tr])
        m = fit_gbt(sc.transform(Xv[tr]), y[tr], seed=SEED,
                    max_depth=params[0], min_samples_leaf=params[1],
                    max_iter=params[2], learning_rate=params[3])
        oof[te] = m.predict_proba(sc.transform(Xv[te]))[:, 1]
    r = full_metrics(y, oof); nested[lbl] = r
    print(f'{lbl:11s} nested AUC-PR={r["auc_pr"]:.4f} lift={r["lift_pr"]:.1f}x '
          f'R@5%={r["recall_at_5pct"]:.3f}  (fixed-param: {base[KEYS[lbl]]["auc_pr"]:.4f})')


node-only   nested AUC-PR=0.2475 lift=6.6x R@5%=0.221  (fixed-param: 0.2434)
graph-only  nested AUC-PR=0.7249 lift=19.2x R@5%=0.740  (fixed-param: 0.7511)
node+graph  nested AUC-PR=0.6807 lift=18.1x R@5%=0.733  (fixed-param: 0.7540)


In [27]:
!pip install dgl


  Using cached dgl-0.1.3-py3-none-manylinux1_x86_64.whl.metadata (497 bytes)
Using cached dgl-0.1.3-py3-none-manylinux1_x86_64.whl (409 kB)


## 7. Real public benchmarks

**Start with YelpChi.** It is small, downloads automatically, and proves the harness works. T-Finance and DGraph-Fin are where the interesting results are, and where the runtime is most likely to die.

We apply **our** ring-disjoint protocol to **their** data. Published numbers on these datasets use random splits, so our numbers will be lower — that is the point, and it must be stated wherever the comparison appears.


In [21]:
!mkdir -p /content/data

# Download the CARE-GNN fraud datasets
!wget -q --show-progress \
  https://raw.githubusercontent.com/safe-graph/CARE-GNN/master/data/YelpChi.mat \
  -O /content/data/YelpChi.mat

!wget -q --show-progress \
  https://raw.githubusercontent.com/safe-graph/CARE-GNN/master/data/Amazon.mat \
  -O /content/data/Amazon.mat

!ls -lh /content/data

total 4.0K
-rw-r--r-- 1 root root    0 Sep  4 22:21 Amazon.mat
drwxr-xr-x 4 root root 4.0K Sep  4 21:45 jale_colab
-rw-r--r-- 1 root root    0 Sep  4 22:21 YelpChi.mat


In [23]:
!rm -f /content/data/YelpChi.mat /content/data/Amazon.mat
!mkdir -p /content/data

# Download the actual dataset archives
!wget -q --show-progress https://data.dgl.ai/dataset/FraudYelp.zip -O /content/data/YelpChi.zip
!wget -q --show-progress https://data.dgl.ai/dataset/FraudAmazon.zip -O /content/data/Amazon.zip

# Extract them
!unzip -o -q /content/data/YelpChi.zip -d /content/data
!unzip -o -q /content/data/Amazon.zip -d /content/data

# Find what was actually extracted
!find /content/data -maxdepth 3 -type f | grep -Ei 'YelpChi|Amazon'

/content/data/YelpC 100%[===================>]  17.15M  --.-KB/s    in 0.1s    
/content/data/Amazo 100%[===================>]  24.91M  16.9MB/s    in 1.5s    
/content/data/Amazon.zip
/content/data/YelpChi.zip
/content/data/Amazon.mat
/content/data/YelpChi.mat


In [24]:
!ls -lh /content/data/YelpChi.mat /content/data/Amazon.mat

-rw-rw-r-- 1 root root 213M Aug 20  2020 /content/data/Amazon.mat
-rw-rw-r-- 1 root root 199M Aug 20  2020 /content/data/YelpChi.mat


In [25]:
from scipy.io import loadmat

for f in ["/content/data/YelpChi.mat", "/content/data/Amazon.mat"]:
    m = loadmat(f)
    print("\n", f)
    print([k for k in m.keys() if not k.startswith("__")])


 /content/data/YelpChi.mat
['homo', 'net_rur', 'net_rtr', 'net_rsr', 'features', 'label']

 /content/data/Amazon.mat
['homo', 'net_upu', 'net_usu', 'net_uvu', 'features', 'label']


In [30]:
import sys
sys.modules.pop("jale.data.public_datasets", None)

from jale.data.public_datasets import load_yelpchi

g = load_yelpchi()
print(g.summary())

YelpChi: 45,954 nodes, 32 features, 6,677 positive (14.53%), 7,693,958 directed edges, avg degree 167.4


In [31]:
from jale.data.public_datasets import (load_yelpchi, load_amazon, load_bwgnn_pt,
                                       load_dgraphfin, subsample,
                                       ring_disjoint_folds_from_adjacency)

g = load_yelpchi()          # ~200 MB, fits free Colab
print(g.summary())
g = subsample(g, 0.25)      # drop to 25% if memory is tight
print(g.summary())
gfolds = ring_disjoint_folds_from_adjacency(g.A, n_splits=N_SPLITS)
print('fold sizes:', np.bincount(gfolds))


YelpChi: 45,954 nodes, 32 features, 6,677 positive (14.53%), 7,693,958 directed edges, avg degree 167.4
YelpChi[sub 25%]: 16,496 nodes, 32 features, 6,677 positive (40.48%), 990,262 directed edges, avg degree 60.0
  WARNING 98.8% of nodes are in one connected component; component-wise folding would collapse to 5 near-empty folds.
  Falling back to degree-bucket grouping. This is a weaker guarantee and must be disclosed wherever the number is reported.
fold sizes: [8643 5046 1812  477  518]


In [32]:
# Same ring-disjoint CV, now on real data. sklearn first (fast, always works).
Xraw_r = np.nan_to_num(g.X, nan=0.0, posinf=0.0, neginf=0.0)
yr = g.y.astype(int)
oof = np.zeros(len(yr))
for tr, te in GroupKFold(n_splits=N_SPLITS).split(Xraw_r, yr, groups=gfolds):
    sc = StandardScaler().fit(Xraw_r[tr])      # fitted on training rows only
    Xtr, Xte = sc.transform(Xraw_r[tr]), sc.transform(Xraw_r[te])
    oof[te] = fit_gbt(Xtr, yr[tr], seed=SEED).predict_proba(Xte)[:, 1]
m = full_metrics(yr, oof)
print(f'{g.name} ring-disjoint  AUC-PR={m["auc_pr"]:.4f} AUC-ROC={m["auc_roc"]:.4f} R@5%={m["recall_at_5pct"]:.3f}')

# Compare against the SAME model on a random split of the same data.
oof2 = np.zeros(len(yr))
for tr, te in StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED).split(Xraw_r, yr):
    sc = StandardScaler().fit(Xraw_r[tr])
    oof2[te] = fit_gbt(sc.transform(Xraw_r[tr]), yr[tr], seed=SEED).predict_proba(
        sc.transform(Xraw_r[te]))[:, 1]
m2 = full_metrics(yr, oof2)
print(f'{g.name} random split    AUC-PR={m2["auc_pr"]:.4f} AUC-ROC={m2["auc_roc"]:.4f}')
print(f'gap on REAL data          = {m2["auc_pr"] - m["auc_pr"]:+.4f}')


YelpChi[sub 25%] ring-disjoint  AUC-PR=0.8868 AUC-ROC=0.9173 R@5%=0.122
YelpChi[sub 25%] random split    AUC-PR=0.9089 AUC-ROC=0.9335
gap on REAL data          = +0.0221


In [35]:
data = to_pyg_data(
    g.A,
    Xr.astype(np.float32),
    yr.astype(np.int64),
    gfolds != 0,
    gfolds == 1,
    gfolds == 2
)

for kind in ['sage', 'caregnn', 'bwgnn']:
    model = build_model(
        kind,
        in_dim=Xr.shape[1],
        hidden=128
    )

In [36]:
# GNN on real data -- only after section 5 runs clean.
data = to_pyg_data(g.A, Xr.astype(np.float32), yr.astype(np.int64),
                   gfolds != 0, gfolds == 1, gfolds == 2)
for kind in ['sage', 'caregnn', 'bwgnn']:
    model = build_model(kind, in_dim=Xr.shape[1], hidden=128)
    _, met = train_and_eval(model, data, epochs=200, device=DEVICE)
    print(f'{g.name} {kind:9s} AUC-PR={met["auc_pr"]:.4f} AUC-ROC={met["auc_roc"]:.4f}')


YelpChi[sub 25%] sage      AUC-PR=0.8932 AUC-ROC=0.9230
YelpChi[sub 25%] caregnn   AUC-PR=0.7686 AUC-ROC=0.8343
YelpChi[sub 25%] bwgnn     AUC-PR=0.4251 AUC-ROC=0.5241


In [ ]:
# Larger benchmarks -- uncomment one at a time. GPU runtime required.
# g = load_bwgnn_pt('/content/data/T-Finance.pt', 'T-Finance'); print(g.summary())
# g = load_dgraphfin('/content/data/dgraphfin.npz');           print(g.summary())
# g = subsample(g, 0.10)   # DGraph-Fin has 3.7M nodes; subsample hard


## 8. Ablations — the experiments that actually convince a reviewer

Run these. They are more persuasive than any single headline number.


In [37]:
# (a) Per-relation ablation: which relations carry the signal?
for rel in graph.relations():
    keep = [c for c in G_COLS if f'_{rel}' not in c and not c.startswith(f'{rel}_')]
    Xa = X_node.join(gf.reindex(graph.app_ids)[keep], how='left')
    m = full_metrics(y, cv(Xa, y, groups, 'gbt'))
    print(f'drop {rel:10s} -> AUC-PR={m["auc_pr"]:.4f}  (full={base["node+graph GBT"]["auc_pr"]:.4f})')


drop device     -> AUC-PR=0.5494  (full=0.7540)
drop dealer     -> AUC-PR=0.6558  (full=0.7540)
drop account    -> AUC-PR=0.5374  (full=0.7540)
drop person     -> AUC-PR=0.7407  (full=0.7540)
drop guarantor  -> AUC-PR=0.6735  (full=0.7540)


In [38]:
# (b) Train on seen typologies, test on an unseen one.
#     Does the model learn RINGS, or does it learn THESE rings?
rings = pd.read_parquet(root/'labels'/'rings.parquet')
appl  = pd.read_parquet(root/'labels'/'application_labels.parquet')
typ   = appl.merge(rings[['ring_id','typology']], on='ring_id', how='left')
typ   = typ.set_index('application_id')['typology'].reindex(graph.app_ids)

for held in sorted(typ.dropna().unique()):
    te = (typ == held).to_numpy()
    tr = ~te
    Xtr, Xte = prep(X_all, tr)[tr], prep(X_all, tr)[te]
    s = fit_gbt(Xtr, y[tr], seed=SEED).predict_proba(Xte)[:, 1]
    mm = full_metrics(y[te], s)
    print(f'held out {held:18s} n={te.sum():4d} pos={int(y[te].sum()):3d} '
          f'AUC-PR={mm.get("auc_pr", float("nan")):.4f}')


held out dealer_collusion   n=  34 pos= 34 AUC-PR=nan
held out device_farm        n=  22 pos= 22 AUC-PR=nan
held out disbursement_sink  n=  25 pos= 25 AUC-PR=nan
held out guarantor_star     n=  39 pos= 39 AUC-PR=nan
held out identity_reuse     n=  11 pos= 11 AUC-PR=nan


## 9. Summary

Copy this table into the report. Report the **ring-disjoint** numbers as the headline. The random-split numbers exist only to quantify the leakage.


In [40]:
rows = [('sklearn ' + k, v['auc_pr'], v['auc_roc']) for k, v in base.items()]
rows += [('GNN ' + k, v['auc_pr'], v['auc_roc']) for k, v in results_gnn.items()]
summary = pd.DataFrame(rows, columns=['model', 'auc_pr', 'auc_roc']).sort_values('auc_pr', ascending=False)
display(summary)
summary.to_csv('colab_summary.csv', index=False)


,model,auc_pr,auc_roc
3,sklearn node+graph GBT,0.753980,0.979798
2,sklearn graph-only GBT,0.751070,0.982123
5,GNN sage,0.687973,0.828498
6,GNN gat,0.671641,0.850536
7,GNN gcn,0.670105,0.804593
8,GNN caregnn,0.669906,0.876543
4,sklearn node+graph graph-LR,0.422553,0.730130
1,sklearn node-only GBT,0.243417,0.570462
0,sklearn node-only LR,0.144000,0.670415
9,GNN bwgnn,0.072287,0.367775


## Known open items

1. **Node-only LR anomaly.** Scores 0.144 AUC-PR (3.8x lift) while node-only GBT scores 0.038 (chance). Unresolved. Hypothesis: the linear model extrapolates past the training range on 11 unbounded numeric features with only 131 positives.
2. **`torch_gnn.py` is unexecuted.** BWGNN's Chebyshev coefficients use a DCT-II that should be checked against the authors' `BetaWavelet.get_filter`. CARE-GNN here is a simplification of the published algorithm, not the algorithm.
3. **L4 ring-level scoring is not implemented.** The plan describes it; the code does not do it. Every number here is still node-level.
4. **No explanation module (L5).** Required for the live demo.
5. **`load_amazon` uses PyG's `Amazon(name='Computers')`**, which is a co-purchase graph, not the fraud graph used in the anomaly-detection literature. Verify which one you actually loaded before quoting a number against published Amazon results.
